# Clustering Digits

In [1]:
import torch
from torchvision import datasets, transforms, utils
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

In [2]:
from torch.utils.data import DataLoader
from torch.optim import Adam

In [3]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from tqdm import tqdm
from sklearn.cluster import KMeans
import pickle

In [4]:
from os.path import exists, join as path_join
from os import makedirs

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
device

device(type='cuda')

In [7]:
emnist_root = 'datasets/emnist'

In [8]:
# Define transformations
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # ResNet expects 3 channels
    transforms.Resize((32, 32)),  # Resize to 32x32 as ResNet typically expects this size
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [9]:
datasets.EMNIST.url ="https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip"

In [10]:
print(f"EMNIST splits: {datasets.EMNIST.splits}")

EMNIST splits: ('byclass', 'bymerge', 'balanced', 'letters', 'digits', 'mnist')


## Clustering letters

In [11]:
emnist_split = 'letters'

In [12]:
exps_root = f"clustering_{emnist_split}"
makedirs(exps_root, exist_ok=True)

In [13]:
num_unique_labels = 10

In [14]:
# Load the EMNIST dataset
train_dataset = datasets.EMNIST(
    root=emnist_root,
    split=emnist_split,
    train=True,
    transform=transform,
    download=True
)

In [15]:
test_dataset = datasets.EMNIST(
    root=emnist_root,
    split=emnist_split,  # Use the same split as for the train dataset
    train=False,
    transform=transform,
)

In [16]:
len(test_dataset)

20800

In [17]:
def imshow(img, title):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (2, 1, 0)), cmap='gray')
    plt.title(title)
    plt.show()

### Model definition

In [18]:
# Load a pre-trained ResNet model
class ResNetForMNIST(nn.Module):
    def __init__(self, num_classes=num_unique_labels):
        super(ResNetForMNIST, self).__init__()
        self.resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.resnet.fc = nn.Identity()  # Remove the final fully connected layer

    def forward(self, x):
        return self.resnet(x)

In [19]:
def eval_model(model, data_loader):
    model.eval()
    with torch.no_grad():  # Disable gradient computation for evaluation
        mean_valid_loss = 0
        total = 0
        correct = 0
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to GPU
            features = model(images)
            mean_valid_loss += criterion(classification_layer(features), labels)
            outputs = classification_layer(features)
            _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        mean_valid_loss /= total
        acc = correct/total
        
    return acc, mean_valid_loss

In [20]:
# Initialize the model
model = ResNetForMNIST().to(device)

# Define the final classification layer (for training purposes)
classification_layer = nn.Linear(512, num_unique_labels).to(device)  # 512 is the output size of resnet18's penultimate layer

RuntimeError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Prepare DataLoader with the transformation applied
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Prepare DataLoader for evaluation training acc and loss
valid_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = Adam(list(model.parameters()) + list(classification_layer.parameters()), lr=0.001)


### Training 
If there are pre-computed weights, use them. Otherwise, train

In [ ]:
if not exists(f'resnet_{emnist_split}.pth'):
    # Training loop
    eval_step= 6000
    num_epochs = 10
    for epoch in tqdm(range(num_epochs)):
        for i, (images, labels) in enumerate(train_loader):
            model.train()
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            features = model(images)
            outputs = classification_layer(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            if i % eval_step == 0:
                eval_acc, eval_loss = eval_model(model, valid_loader)
                test_acc, test_loss = eval_model(model, test_loader)
                print(f"[Train | Test] acc: {eval_acc * 100:.2f} | {test_acc * 100:.2f}, loss {eval_loss:.6f} | {test_loss:.6f}")
        print(f"Epoch {epoch+1}, Loss: {loss.item()}")


In [ ]:
# Save the model's state dictionary
if (not exists(path_join(exps_root, f'resnet_{emnist_split}.pth'))) or (not exists(path_join(exps_root, f'resnet_{emnist_split}.pth'))):
    torch.save(model.state_dict(), path_join(exps_root, f'resnet_{emnist_split}.pth'))
    torch.save(classification_layer.state_dict(), path_join(exps_root, f'resnet_class_layer_{emnist_split}.pth'))

In [ ]:
# Load the saved state dictionary
if exists(path_join(exps_root, f'resnet_{emnist_split}.pth')):
    model.load_state_dict(torch.load(path_join(exps_root, f'resnet_{emnist_split}.pth')))
    classification_layer.load_state_dict(torch.load(path_join(exps_root, f'resnet_class_layer_{emnist_split}.pth')))

In [ ]:
model.eval()  # Set the model to evaluation mode

correct = 0
total = 0

model.eval()
with torch.no_grad():  # Disable gradient computation for evaluation
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)  # Move data to GPU
        features = model(images)
        outputs = classification_layer(features)
        _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate accuracy
accuracy = correct / total
print(f'Accuracy of the model on the test set: {accuracy * 100:.2f}%')

In [ ]:
# Extract latent space for clustering
def get_latent_space(model, dataloader):
    model.eval()
    latents = []
    labels = []
    with torch.no_grad():
        for images, _labels in dataloader:
            images = images.to(device)
            features = model(images)
            latents.append(features.cpu())  # Move to CPU for clustering
            labels.append(_labels)
    return torch.cat(latents), torch.cat(labels)


In [ ]:
# Function to extract latent features, grouped by digit class
def get_latent_space_by_class(model, dataloader):
    model.eval()
    latent_dict = {i: [] for i in range(num_unique_labels)}  # Dictionary to store latents for each digit
    label_dict = {i: [] for i in range(num_unique_labels)}   # Dictionary to store corresponding labels

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            features = model(images)
            features = features.cpu()

            for i in range(num_unique_labels):
                mask = (labels == i)
                latent_dict[i].append(features[mask])
                label_dict[i].append(labels[mask])

    # Concatenate all the latent features and labels for each digit
    for i in range(10):
        if latent_dict[i]:
            latent_dict[i] = torch.cat(latent_dict[i])
            label_dict[i] = torch.cat(label_dict[i])
    
    return latent_dict, label_dict

# Extract latent space representations grouped by digit class
latent_dict, label_dict = get_latent_space_by_class(model, train_loader)

In [ ]:
# Perform clustering for each digit class
cluster_assignments = {}
clustering = {}
for i in range(num_unique_labels):
    if latent_dict[i].size(0) > 0:
        clustering[i] = KMeans(n_clusters=2, random_state=0, n_init='auto')
        cluster_assignments[i] = clustering[i].fit_predict(latent_dict[i])

In [ ]:
# Example of how to use the cluster assignments
for i in range(num_unique_labels):
    clusters = cluster_assignments[i]
    labels = label_dict[i]
    count_0 = (clusters == 0).sum()
    count_1 = (clusters == 1).sum()
    print(f"Digit {i}: Cluster 0 count = {count_0}, Cluster 1 count = {count_1} [{100*count_0/(count_0+count_1):.2f}%]")

In [ ]:
test_latent_dict, test_label_dict = get_latent_space_by_class(model, test_loader)

test_cluster_assignments = {}
for i in range(num_unique_labels):
    if latent_dict[i].size(0) > 0:
        test_cluster_assignments[i] = clustering[i].predict(test_latent_dict[i])

In [ ]:
if exists(path_join(exps_root, f'kmeans_{emnist_split}_clusterings.pkl')):
    with open(path_join(exps_root, f'kmeans_{emnist_split}_clusterings.pkl'), 'rb') as f:
        clustering = pickle.load(f)

if exists(path_join(exps_root, f'kmeans_{emnist_split}_train_clusters.pkl')):
    with open(path_join(exps_root, f'kmeans_{emnist_split}_train_clusters.pkl'), 'rb') as f:
        cluster_assignments = pickle.load(f)

if exists(path_join(exps_root, f'kmeans_{emnist_split}_test_clusters.pkl')):
    with open(path_join(exps_root, f'kmeans_{emnist_split}_test_clusters.pkl'), 'rb') as f:
        test_cluster_assignments = pickle.load(f)

In [ ]:
# Example of how to use the cluster assignments
for i in range(num_unique_labels):
    test_clusters = test_cluster_assignments[i]
    test_labels = test_label_dict[i]
    count_0 = (test_clusters == 0).sum()
    count_1 = (test_clusters == 1).sum()
    print(f"Digit {i}: Cluster 0 count = {count_0}, Cluster 1 count = {count_1} [{100*count_0/(count_0+count_1):.2f}%]")

In [ ]:
# Save the KMeans models
if not exists(path_join(exps_root, f'kmeans_{emnist_split}_clusterings.pkl')):
    with open(path_join(exps_root, f'kmeans_{emnist_split}_clusterings.pkl'), 'wb') as f:
        pickle.dump(clustering, f)
if not exists(path_join(exps_root, f'kmeans_{emnist_split}_train_clusters.pkl')):
    with open(path_join(exps_root, f'kmeans_{emnist_split}_train_clusters.pkl'), 'wb') as f:
        pickle.dump(cluster_assignments, f)
if not exists(path_join(exps_root, f'kmeans_{emnist_split}_test_clusters.pkl')):
    with open(path_join(exps_root, f'kmeans_{emnist_split}_test_clusters.pkl'), 'wb') as f:
        pickle.dump(test_cluster_assignments, f)


In [ ]:
from sklearn.manifold import TSNE

In [ ]:
if not exists(f'tsne_{emnist_split}_train_model.pkl'):
    tsne_dict = {}
    tsne_coords_dict = {}
    for i in tqdm(range(num_unique_labels)):
        tsne_dict[i] = TSNE(n_components=2, random_state=0)
        tsne_coords_dict[i] = tsne_dict[i].fit_transform(latent_dict[i])
        
    with open(path_join(exps_root, f'tsne_{emnist_split}_train_model.pkl'), 'wb') as f:
        pickle.dump(tsne_dict, f)
    with open(path_join(exps_root, f'tsne_{emnist_split}_train_coords.pkl'), 'wb') as f:
        pickle.dump(tsne_coords_dict, f)
else:
    with open(path_join(exps_root, f'tsne_{emnist_split}_train_model.pkl'), 'rb') as f:
        tsne_dict = pickle.load(f)
    with open(path_join(exps_root, f'tsne_{emnist_split}_train_coords.pkl'), 'rb') as f:
        tsne_coords_dict = pickle.load(f)

In [ ]:
fig = plt.figure()

fig.set_figheight(10)
fig.set_figwidth(15)
cols = 3
rows = num_unique_labels//cols + (num_unique_labels/cols > 0)

for i in range(num_unique_labels):
    ax = fig.add_subplot(cols,rows,i+1)
    ax.axis('off')
    ax.set_title(f"'{train_dataset.classes[i]}' char")
    plt.scatter(tsne_coords_dict[i][:,0], tsne_coords_dict[i][:,1], c=cluster_assignments[i], cmap='tab10')

# plt.subplots_adjust(wspace=0.4, hspace=0.4)
plt.tight_layout()
plt.subplots_adjust(top=0.90)
plt.suptitle(f"TSNE projection and Clustering of {emnist_split} characters", fontsize=20)
plt.show()

In [ ]:
tsne_coords_dict[0].shape

In [ ]:
from itertools import chain

In [ ]:
latent_dict_all = np.vstack(list(chain([latent_dict[i] for i in range(num_unique_labels)])))
cluster_assignments_all_ = np.hstack(list(chain([cluster_assignments[i] for i in range(num_unique_labels)])))
_tmp_class_assignments_all = np.hstack(list(chain([[i]*len(cluster_assignments[i]) for i in range(num_unique_labels)])))

class_assignments_all = np.vstack([_tmp_class_assignments_all, cluster_assignments_all_]).T

In [ ]:
latent_dict_all.shape, cluster_assignments_all_.shape, _tmp_class_assignments_all.shape, class_assignments_all.shape

In [ ]:
class_assignments_all[:, 0:6]

class_assignments_all_str = np.core.defchararray.add(class_assignments_all.astype(str)[:, 0], ' ')
class_assignments_all_str = np.core.defchararray.add(class_assignments_all_str, class_assignments_all.astype(str)[:, 1])

unique_class_assignments_all_str = np.unique(class_assignments_all_str)
color_map = {string: color for string, color in zip(unique_class_assignments_all_str, matplotlib.colormaps.get_cmap('tab20')(range(len(unique_class_assignments_all_str))))}

# Map each entry in merged_array to a color
colors = np.array([color_map[string] for string in class_assignments_all_str])


In [ ]:
tsne_all = TSNE(n_components=2, random_state=0)
tsne_coords_all = tsne_all.fit_transform(latent_dict_all)

In [ ]:
with open(path_join(exps_root, f'tsne_{emnist_split}_all_train_model.pkl'), 'wb') as f:
    pickle.dump(tsne_all, f)
with open(path_join(exps_root, f'tsne_{emnist_split}_all_train_coords.pkl'), 'wb') as f:
    pickle.dump(tsne_coords_all, f)
with open(path_join(exps_root, f'kmeans_{emnist_split}_all_train_clusters.pkl)', 'wb') as f:
    pickle.dump(class_assignments_all, f)

In [ ]:
if exists(path_join(exps_root, f'tsne_{emnist_split}_all_train_model.pkl')):
    with open(path_join(exps_root, f'tsne_{emnist_split}_all_train_model.pkl'), 'rb') as f:
        tsne_all = pickle.load(f)
if exists(path_join(exps_root, f'tsne_{emnist_split}_all_train_coords.pkl'),):
    with open(path_join(exps_root, f'tsne_{emnist_split}_all_train_coords.pkl'), 'rb') as f:
        tsne_coords_all = pickle.load(f)
if exists(path_join(exps_root, f'kmeans_{emnist_split}_all_train_clusters.pkl')):
    with open(path_join(exps_root, f'kmeans_{emnist_split}_all_train_clusters.pkl'), 'rb') as f:
        class_assignments_all = pickle.load(f)

In [ ]:
fig = plt.figure()

fig.set_figheight(10)
fig.set_figwidth(15)

plt.scatter(tsne_coords_all[:,0], tsne_coords_all[:,1], c=colors, cmap='tab10')
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[string], markersize=8, label=string) for string in unique_class_assignments_all_str]
plt.legend(handles=handles, title="Clusters")
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
train_dataset

In [ ]:
class ClusteredEMNIST(torch.utils.data.Dataset):
    def __init__(self, original_dataset, encoder, clustering, num_unique_labels, batch_size=64):
        self.original_dataset = original_dataset
        self.sub_class = self._generate_new_column(original_dataset, encoder, clustering, num_unique_labels, batch_size)
        assert len(original_dataset), len(self.sub_class)
        
    def __len__(self):
        return len(self.original_dataset)

    def __getitem__(self, index):
        # Get the original data and target
        img, target = self.original_dataset[index]
        sub_class = self.sub_class[index]  # You can modify this logic as needed
        
        # Step 5: Return a tuple with the original data and the new column
        return img, target, sub_class
        
    def _generate_new_column(self, original_dataset, encoder, clustering, num_unique_labels, batch_size=64):
        # Prepare DataLoader
        dataloader = DataLoader(original_dataset, batch_size=batch_size, shuffle=False, num_workers=1)
    
        new_column = []
        
        # Step 1: Process data in batches
        digit_tensors = {digit: [] for digit in range(num_unique_labels)}
        digit_indices = {digit: [] for digit in range(num_unique_labels)}
    
        for batch_idx, (imgs, targets) in enumerate(dataloader):
            imgs = imgs.to(device)
            features = encoder(imgs)  # Assume encoder is a model that extracts features
    
            for i, target in enumerate(targets):
                target = target.item()
                digit_tensors[int(target)].append(features[i].cpu())
                digit_indices[int(target)].append(batch_idx * batch_size + i)
    
        # Step 2: Predict new_column for each digit using the clustering model
        for digit in range(num_unique_labels):
            if digit_tensors[digit]:
                # Stack the images to pass through the model
                tensor_stack = torch.stack(digit_tensors[digit])
                
                # Predict new_column values using the clustering model
                new_column_values = clustering[digit].predict(tensor_stack.cpu().detach().numpy().astype('float'))
                
                # Assign the predicted values to the corresponding indices
                for i, idx in enumerate(digit_indices[digit]):
                    new_column.append((idx, new_column_values[i].item()))
        
        # Step 3: Sort the new_column list by the original index order
        new_column.sort(key=lambda x: x[0])
        
        # Extract just the new column values in order
        new_column = [val for _, val in new_column]
        
        return new_column

In [ ]:
# Step 6: Instantiate the custom dataset
clustered_train_dataset = ClusteredEMNIST(train_dataset,
                                          model, clustering, num_unique_labels, batch_size=64)
clustered_test_dataset = ClusteredEMNIST(test_dataset,
                                          model, clustering, num_unique_labels, batch_size=64)

In [ ]:
# Step 7: Access the data
img, target, sub_class = clustered_train_dataset[0]
print(f"Image shape: {img.shape}, Target: {target}, Sub-class: {sub_class}")

In [ ]:
count = {}
for img, target, sub_class in clustered_train_dataset:
    count[target] = count.get(target, {0:0, 1:0})
    count[target][sub_class] += 1
    count[target]["%"] = count[target][0]/(count[target][0]+count[target][1])
print(count)

In [ ]:
assert count[0][0] == 4018 and count[0][1] == 1982
assert count[4][0] == 3897 and count[4][1] == 2103

In [ ]:
count_test = {}
for img, target, sub_class in clustered_test_dataset:
    count_test[target] = count_test.get(target, {0:0, 1:0})
    count_test[target][sub_class] += 1
    count_test[target]["%"] = count_test[target][0]/(count_test[target][0]+count_test[target][1])
print(count_test)

In [ ]:
assert count_test[0][0] == 651 and count_test[0][1] == 349, f"{count_test[0][0]} == 651 and {count_test[0][1]} == 349"
assert count_test[4][0] == 649 and count_test[4][1] == 351, f"{count_test[4][0]} == 649 and {count_test[4][1]} == 351"

In [ ]:
print(train_train_dataset[0])

In [ ]:
len(train_dataset), len(l)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
if not exists(path_join(exps_root, f"train_{emnist_split}_dataset.pkl")):
    with open(path_join(exps_root, f"train_{emnist_split}_dataset.pkl"), 'wb') as f:
        pickle.dump(clustered_train_dataset, f)

if not exists(path_join(exps_root, f"test_{emnist_split}_dataset.pkl")):
    with open(path_join(exps_root, f"test_{emnist_split}_dataset.pkl"), 'wb') as f:
        pickle.dump(clustered_test_dataset, f)
